In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import numpy as np
import pandas as pd
### Попередній код з кластерізації
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
# Використовуємо відфільтрований датафрейм (без tvod == 1)
user_df = pd.read_csv(r'C:\Users\pinko\Documents\dell files\desktop\analytical project\data\user_df_sample.csv')
cluster_df = user_df.drop(columns=['IOS']).copy()
#filtered_df = user_df[user_df['tvod'] != 1].copy()

# Обираємо числові ознаки для кластеризації
# (Виключаємо ID та константні колонки, де лише одне унікальне значення)
features = cluster_df.select_dtypes(include=['number']).copy()
features = features.loc[:, features.nunique() > 1]

# Заповнюємо можливі пропуски нулями
features = features.fillna(0)


# Стандартизація даних

features_log = np.log1p(features.clip(lower=0))
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features_log)

OPTIMAL_K = 4
kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
cluster_df['cluster'] = kmeans.fit_predict(X_scaled)

# 1. Беремо 'features', на яких робили кластеризацію
X = features.copy()
y = cluster_df['cluster']

# 2. Розділяємо дані на навчальну та тестову вибірки
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Навчаємо Random Forest

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# 4. Перевіряємо, наскільки точно модель відтворює логіку кластерів
y_pred = rf_model.predict(X_test)
print(f"Точність моделі (Accuracy): {accuracy_score(y_test, y_pred):.2%}\n")
print(classification_report(y_test, y_pred))

# 5. Будуємо графік Feature Importance
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
top_features = importances.sort_values(ascending=True).tail(15) # беремо топ-15

top15 = importances.sort_values(ascending=False).head(15)

# тільки назви фіч
top_features_list = top15.index.to_list()
print(top_features_list)

# назви + значення (у %), зручно для README
print((top15 * 100).round(2))

plt.figure(figsize=(10, 7))
top_features.plot(kind='barh', color='#2a9d8f')
plt.title('Топ-15 найважливіших ознак для поділу на кластери (Random Forest)', fontsize=13)
plt.xlabel('Важливість ознаки (Feature Importance)')
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.savefig(r'C:\Users\pinko\Documents\dell files\desktop\analytical project\images\feature_importance.png', dpi=150, bbox_inches='tight')
plt.tight_layout()
plt.show()
